# Data Wrangling: Walkability

The next dataset I’m using for the opportunity finder is walkability. Walkability matters because it gives an idea of how easy it is to move around without relying only on a car. For this project, it could help show which tracts already have strong built environment conditions and which areas might need better streets, land use mix, or access.

I’m using the EPA National Walkability Index because it already includes walkability-related fields. The data starts at the block group level, so I’ll roll it up to census tracts later.

In [50]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [7]:
# set project paths
raw_dir = Path('../data/raw/WalkabilityIndex')
processed_dir = Path('../data/processed')

# update this once the file is downloaded
walkability_path = raw_dir / 'Natl_WI.gdb'

output_path = processed_dir / 'walkability_by_tract.csv'

In [8]:
import fiona

# see what layers are inside the geodatabase
layers = fiona.listlayers(walkability_path)

layers

['NationalWalkabilityIndex']

## Check walkability data

The EPA file is stored as a geodatabase, and found that there's only one layer called `NationalWalkabilityIndex`.

After loading that latyer, I can inspect the columns, missing values, CRS, and geometry.

In [9]:
# load walkability layer
walk_raw = gpd.read_file(walkability_path, layer='NationalWalkabilityIndex')

walk_raw.head()

,GEOID10,GEOID20,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,CSA,CSA_Name,CBSA,CBSA_Name,Ac_Total,Ac_Water,Ac_Land,Ac_Unpr,TotPop,CountHU,HH,Workers,D2B_E8MIXA,D2A_EPHHM,D3B,D4A,D2A_Ranked,D2B_Ranked,D3B_Ranked,D4A_Ranked,NatWalkInd,Shape_Length,Shape_Area,geometry
0,481130078254,481130078254,48,113,007825,4,206,"Dallas-Fort Worth, TX-OK",19100,"Dallas-Fort Worth-Arlington, TX",73.595028,0.0,73.595028,73.595028,1202,460.0,423.0,412,0.662091,0.348912,115.981747,362.10,6.0,14.0,15.0,17.0,14.000000,3110.360820,297836.083090,"MULTIPOLYGON (((-68983.316 1091325.734, -68981..."
1,481130078252,481130078252,48,113,007825,2,206,"Dallas-Fort Worth, TX-OK",19100,"Dallas-Fort Worth-Arlington, TX",119.829909,0.0,119.829909,119.214200,710,409.0,409.0,395,0.554458,0.197047,80.145600,718.84,3.0,10.0,12.0,14.0,10.833333,3519.469110,484945.146563,"MULTIPOLYGON (((-68891.713 1090955.557, -68860..."
2,481130078253,481130078253,48,113,007825,3,206,"Dallas-Fort Worth, TX-OK",19100,"Dallas-Fort Worth-Arlington, TX",26.367053,0.0,26.367053,26.367050,737,365.0,329.0,463,-0.000000,0.000000,24.272717,398.31,1.0,1.0,7.0,17.0,8.333333,1697.091802,106705.928129,"MULTIPOLYGON (((-68078.32 1091181.799, -68077...."
3,481130078241,481130078241,48,113,007824,1,206,"Dallas-Fort Worth, TX-OK",19100,"Dallas-Fort Worth-Arlington, TX",119.060687,0.0,119.060687,119.060687,904,384.0,384.0,431,0.553831,0.682830,141.604424,386.24,16.0,10.0,17.0,17.0,15.666667,2922.609204,481828.430336,"MULTIPOLYGON (((-68978.261 1090638.77, -68976...."
4,481130078242,481130078242,48,113,007824,2,206,"Dallas-Fort Worth, TX-OK",19100,"Dallas-Fort Worth-Arlington, TX",169.927211,0.0,169.927211,148.742920,948,343.0,343.0,579,0.459064,0.261472,65.307963,638.37,4.0,7.0,11.0,14.0,10.166667,3731.971773,687684.775181,"MULTIPOLYGON (((-68980.363 1090202.6, -68965.2..."


In [15]:
walk_raw.shape

(220739, 30)

In [16]:
walk_raw.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 220739 entries, 0 to 220738
Data columns (total 30 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   GEOID10       220739 non-null  str     
 1   GEOID20       220739 non-null  str     
 2   STATEFP       220739 non-null  str     
 3   COUNTYFP      220739 non-null  str     
 4   TRACTCE       220739 non-null  str     
 5   BLKGRPCE      220739 non-null  str     
 6   CSA           167709 non-null  str     
 7   CSA_Name      167709 non-null  str     
 8   CBSA          220739 non-null  str     
 9   CBSA_Name     203645 non-null  str     
 10  Ac_Total      220739 non-null  float64 
 11  Ac_Water      220739 non-null  float64 
 12  Ac_Land       220739 non-null  float64 
 13  Ac_Unpr       220739 non-null  float64 
 14  TotPop        220739 non-null  int32   
 15  CountHU       220464 non-null  float64 
 16  HH            220464 non-null  float64 
 17  Workers       220739 

In [13]:
# check CRS
walk_raw.crs

<Projected CRS: ESRI:102039>
Name: USA_Contiguous_Albers_Equal_Area_Conic_USGS_version
Axis Info [cartesian]:
- [east]: Easting (metre)
- [north]: Northing (metre)
Area of Use:
- undefined
Coordinate Operation:
- name: unnamed
- method: Albers Equal Area
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [14]:
# check missing values
missing = walk_raw.isna().sum().sort_values(ascending=False)

missing[missing > 0].head(30)

CSA_Name     53030
CSA          53030
CBSA_Name    17094
CountHU        275
HH             275
dtype: int64

## CRS note

The walkability file uses a projected CRS. I do not need to change the CRS yet because I am filtering by ID columns, not doing a spatial join.

If I use this geometry later with tract boundaries or maps, I’ll reproject it to match the other spatial files before doing any spatial analysis.

In [17]:
# filter to San Diego County
walk_sd = walk_raw[
    (walk_raw['STATEFP'] == '06') & # california
    (walk_raw['COUNTYFP'] == '073')].copy() # san diego county

walk_sd.shape

(1795, 30)

In [18]:
walk_sd.head()

,GEOID10,GEOID20,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,CSA,CSA_Name,CBSA,CBSA_Name,Ac_Total,Ac_Water,Ac_Land,Ac_Unpr,TotPop,CountHU,HH,Workers,D2B_E8MIXA,D2A_EPHHM,D3B,D4A,D2A_Ranked,D2B_Ranked,D3B_Ranked,D4A_Ranked,NatWalkInd,Shape_Length,Shape_Area,geometry
31564,060730170341,060730170341,06,073,017034,1,NaN,NaN,41740,"San Diego-Chula Vista-Carlsbad, CA",474.007729,0.0,474.007729,379.373090,2420,917.0,894.0,1156,0.602618,0.440297,49.978594,-99999.00,8.0,12.0,9.0,1.0,6.666667,11690.850709,1.918283e+06,"MULTIPOLYGON (((-1943715.812 1319640.667, -194..."
31565,060730170142,060730170142,06,073,017014,2,NaN,NaN,41740,"San Diego-Chula Vista-Carlsbad, CA",198.966155,0.0,198.966155,198.966155,1229,757.0,732.0,507,0.214579,0.273003,117.985895,929.40,4.0,3.0,16.0,13.0,10.833333,4313.386796,8.052062e+05,"MULTIPOLYGON (((-1940508.543 1320858.495, -194..."
31566,060730171042,060730171042,06,073,017104,2,NaN,NaN,41740,"San Diego-Chula Vista-Carlsbad, CA",110.680538,0.0,110.680538,110.680538,1627,501.0,501.0,740,0.674690,0.402081,84.845269,934.76,7.0,14.0,13.0,13.0,12.166667,3152.599220,4.479185e+05,"MULTIPOLYGON (((-1955787.105 1327950.922, -195..."
31567,060730185042,060730185042,06,073,018504,2,NaN,NaN,41740,"San Diego-Chula Vista-Carlsbad, CA",580.982508,0.0,580.982508,555.437472,1606,621.0,575.0,605,0.636255,0.769620,39.306657,-99999.00,18.0,13.0,8.0,1.0,8.166667,7754.365153,2.351201e+06,"MULTIPOLYGON (((-1961970.686 1345796.545, -196..."
31614,060730075012,060730075012,06,073,007501,2,NaN,NaN,41740,"San Diego-Chula Vista-Carlsbad, CA",24.728285,0.0,24.728285,24.728285,592,301.0,272.0,341,0.233766,0.456707,258.838811,183.73,9.0,3.0,20.0,19.0,15.000000,1301.547218,1.000754e+05,"MULTIPOLYGON (((-1962803.315 1295507.416, -196..."


In [19]:
# create a new tract id column using GEOID20, removing the block number
walk_sd['tract_geoid'] = walk_sd['GEOID20'].str[:11]

walk_sd[['GEOID20', 'tract_geoid', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'BLKGRPCE']].head()

,GEOID20,tract_geoid,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE
31564,060730170341,06073017034,06,073,017034,1
31565,060730170142,06073017014,06,073,017014,2
31566,060730171042,06073017104,06,073,017104,2
31567,060730185042,06073018504,06,073,018504,2
31614,060730075012,06073007501,06,073,007501,2


In [43]:
# check how many block groups are inside each tract
walk_sd.groupby('tract_geoid')['GEOID20'].nunique().sort_values(ascending=False).head()

tract_geoid
06073007600    7
06073009706    7
06073001800    6
06073008507    6
06073017022    6
Name: GEOID20, dtype: int64

In [21]:
walk_cols = [
    'tract_geoid',
    'GEOID20',
    'TotPop',
    'CountHU',
    'HH',
    'Workers',
    'Ac_Total',
    'Ac_Water',
    'Ac_Land',
    'Ac_Unpr',
    'D2A_EPHHM',
    'D2B_E8MIXA',
    'D3B',
    'D4A',
    'D2A_Ranked',
    'D2B_Ranked',
    'D3B_Ranked',
    'D4A_Ranked',
    'NatWalkInd']

walk_clean = walk_sd[walk_cols].copy()

walk_clean.head()

,tract_geoid,GEOID20,TotPop,CountHU,HH,Workers,Ac_Total,Ac_Water,Ac_Land,Ac_Unpr,D2A_EPHHM,D2B_E8MIXA,D3B,D4A,D2A_Ranked,D2B_Ranked,D3B_Ranked,D4A_Ranked,NatWalkInd
31564,06073017034,060730170341,2420,917.0,894.0,1156,474.007729,0.0,474.007729,379.373090,0.440297,0.602618,49.978594,-99999.00,8.0,12.0,9.0,1.0,6.666667
31565,06073017014,060730170142,1229,757.0,732.0,507,198.966155,0.0,198.966155,198.966155,0.273003,0.214579,117.985895,929.40,4.0,3.0,16.0,13.0,10.833333
31566,06073017104,060730171042,1627,501.0,501.0,740,110.680538,0.0,110.680538,110.680538,0.402081,0.674690,84.845269,934.76,7.0,14.0,13.0,13.0,12.166667
31567,06073018504,060730185042,1606,621.0,575.0,605,580.982508,0.0,580.982508,555.437472,0.769620,0.636255,39.306657,-99999.00,18.0,13.0,8.0,1.0,8.166667
31614,06073007501,060730075012,592,301.0,272.0,341,24.728285,0.0,24.728285,24.728285,0.456707,0.233766,258.838811,183.73,9.0,3.0,20.0,19.0,15.000000


In [22]:
# check missing values
walk_clean.isna().sum().sort_values(ascending=False)

tract_geoid    0
GEOID20        0
TotPop         0
CountHU        0
HH             0
Workers        0
Ac_Total       0
Ac_Water       0
Ac_Land        0
Ac_Unpr        0
D2A_EPHHM      0
D2B_E8MIXA     0
D3B            0
D4A            0
D2A_Ranked     0
D2B_Ranked     0
D3B_Ranked     0
D4A_Ranked     0
NatWalkInd     0
dtype: int64

In [23]:
# check value ranges
walk_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
TotPop,1795.0,1840.018384,1626.241389,0.000000,1111.500000,1587.000000,2203.000000,38932.000000
CountHU,1795.0,671.244568,483.587081,0.000000,420.000000,584.000000,782.000000,7630.000000
HH,1795.0,623.387187,447.670566,0.000000,389.000000,542.000000,735.500000,6877.000000
Workers,1795.0,790.741504,530.732526,0.000000,498.500000,698.000000,955.500000,7628.000000
Ac_Total,1795.0,1613.621943,14104.179316,14.881103,76.078507,138.539827,290.734524,431297.430315
Ac_Water,1795.0,112.524057,3849.819422,0.000000,0.000000,0.000000,0.000000,162918.626815
Ac_Land,1795.0,1501.097886,13536.134251,0.000000,75.221070,135.154310,286.723844,431295.543212
Ac_Unpr,1795.0,759.647715,4497.100150,0.000000,72.205850,128.294073,250.070997,132453.398470
D2A_EPHHM,1795.0,0.484055,0.207703,0.000000,0.325663,0.467342,0.648829,0.971371
D2B_E8MIXA,1795.0,0.581132,0.189799,-0.000000,0.465694,0.619269,0.723519,0.950655


In [39]:
# use households as the main weight
walk_clean['hh_weight'] = walk_clean['HH']

# avoid zero weights so it doesn't affect the averages
walk_clean.loc[walk_clean['hh_weight'] <= 0, 'hh_weight'] = np.nan

In [40]:
# defining a helper to average walkability values based on how many households are in each block group

def weighted_avg(group, value_col, weight_col='hh_weight'):
    values = group[value_col]
    weights = group[weight_col]

    valid = values.notna() & weights.notna()

    if valid.sum() == 0:
        return values.mean()

    return np.average(values[valid], weights=weights[valid])

In [41]:
# grouping block groups into tracts and summing the totals, walkability values, and built environment

walkability_by_tract = walk_clean.groupby('tract_geoid').apply(
    lambda group: pd.Series({
        'walk_block_group_count': group['GEOID20'].nunique(),
        'walk_tot_pop': group['TotPop'].sum(),
        'walk_households': group['HH'].sum(),
        'walk_housing_units': group['CountHU'].sum(),
        'walk_workers': group['Workers'].sum(),
        'walk_land_acres': group['Ac_Land'].sum(),

        'nat_walk_index': weighted_avg(group, 'NatWalkInd'),
        'd2a_jobs_housing_mix': weighted_avg(group, 'D2A_EPHHM'),
        'd2b_employment_mix': weighted_avg(group, 'D2B_E8MIXA'),
        'd3b_intersection_density': weighted_avg(group, 'D3B'),
        'd4a_commute_mode_split': weighted_avg(group, 'D4A'),

        'd2a_ranked': weighted_avg(group, 'D2A_Ranked'),
        'd2b_ranked': weighted_avg(group, 'D2B_Ranked'),
        'd3b_ranked': weighted_avg(group, 'D3B_Ranked'),
        'd4a_ranked': weighted_avg(group, 'D4A_Ranked')})).reset_index()

walkability_by_tract.head()

,tract_geoid,walk_block_group_count,walk_tot_pop,walk_households,walk_housing_units,walk_workers,walk_land_acres,nat_walk_index,d2a_jobs_housing_mix,d2b_employment_mix,d3b_intersection_density,d4a_commute_mode_split,d2a_ranked,d2b_ranked,d3b_ranked,d4a_ranked
0,06073000100,2.0,3250.0,1342.0,1387.0,1234.0,380.893530,14.740561,0.306264,0.544148,159.972426,236.994382,5.226528,10.763785,17.556632,18.669896
1,06073000201,1.0,1915.0,1019.0,1120.0,940.0,213.546538,18.000000,0.710940,0.854992,133.885195,194.460000,16.000000,20.000000,17.000000,19.000000
2,06073000202,3.0,4583.0,2340.0,2468.0,2348.0,322.712600,15.573006,0.467238,0.670747,160.306312,298.269098,9.379487,14.026923,17.648718,17.367094
3,06073000300,5.0,5094.0,2843.0,3193.0,2502.0,222.084090,15.526556,0.569315,0.527950,150.211521,217.186753,12.260640,9.776644,16.910306,18.650721
4,06073000400,3.0,3758.0,2024.0,2390.0,2144.0,290.948636,12.902503,0.417204,0.307641,120.998178,370.717530,8.009881,5.488142,15.332016,16.626482


In [42]:
# check tract-level output
walkability_by_tract.shape

(628, 16)

After filtering to San Diego County, I created a tract ID from each block group GEOID. This matters because the EPA walkability data starts at the block group level, but the rest of my project is at the tract level.

Then I kept the main walkability and built environment fields, including the National Walkability Index, land use mix, employment mix, intersection density, and ranked versions of those fields.

Finally, I grouped the block groups into census tracts and used household-weighted averages so areas with more households had more influence on the tract-level walkability values.

## Save cleaned walkability file

The final output is now one row per tract and can be joined with the ACS, crime, and transit files using `tract_geoid`.

In [46]:
walkability_by_tract.to_csv(output_path, index=False)

output_path

WindowsPath('../data/processed/walkability_by_tract.csv')

## Walkability wrangling summary

The original dataset was pretty clean and had all of the info that I needed, so the main thing I had to do was filter it to San Diego, then calculate scores from each of the rows -- the data was broken down to the block level, but since my project is at the tract level, I had to calculate weighted scores and assign them to each tract. 

I created a tract ID using the GEOID20 column, which is the tract plus the block number, then grouped block groups into census tracts, and summarized the walkability fields.

For totals like population, households, workers, and land acres, I added the values together. For walkability fields, I used household-weighted averages so block groups with more households had stronger influence. 